## **Finetuning Qwen3 8B with Unsloth**


LLM: given tokens -> predict next token
Fine-tuning: Change the weights of how LLM predict next token by providing it examples of "this text" -> "followed by this text"

1. Load source/target pairs
2. Turn each pair into a long string which is how Qwen sees it
3. Tell trainer to only learn from brainrot answer part (second part)
4. Run training loop
5. Save adapter

In [ ]:
%%capture
# Installs Unsloth, Xformers (Flash Attention), and all other packages!
%pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
# import models
from unsloth import FastLanguageModel, FastModel
from unsloth.chat_templates import train_on_responses_only

import torch

from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

from shared import SYSTEM_PROMPT

In [ ]:
# load dataset
ds = load_dataset("json", data_files={"train": "data/train.jsonl", "val": "data/val.jsonl"})


In [ ]:
# format: list of 3 dicts is standard format openAI, Gemini, and others use
# Given row, format a three-msg conversation
#   system: SYSTEM_PROMPT
#   user: row["source"] Ex. "Why am I like this?"
#   assistant: row["target"] Ex. "system error: brain.exe encountered feelings again"
# tokenizer.apply_chat_template() format into raw string Qwen was trained on
def format(row): # given a row, format such that
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["source"]},
            {"role": "assistant", "content": row["target"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, enable_thinking=False)}

In [ ]:




max_seq_length = 2048 # Supports RoPE Scaling internally, so choose any!

# Get brainrot dataset
url = "https://huggingface.co/datasets/ethan00alphayehah/brainrot-dataset"
dataset = load_dataset("json", data_files = {"train" : url}, split = "train")

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/qwen3-8b-unsloth-bnb-4bit", #or choose any model
]


# Load model
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/qwen3-8b",
    max_seq_length = 256, # Choose any for long context!
    load_in_4bit = True,  # 4-bit quantization. False = 16-bit LoRA.
    load_in_8bit = False, # 8-bit quantization
    load_in_16bit = False, # [NEW!] 16-bit LoRA
    full_finetuning = False, # Use for full fine-tuning.
    # token = "hf_...", # use one if using gated models
)

# Attach LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    max_seq_length = max_seq_length,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

# train model with LoRA
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    tokenizer = tokenizer,
    args = SFTConfig(
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 60,
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
        seed = 3407,
    ),
)
trainer.train()

# Go to https://docs.unsloth.ai for advanced tips like
# (1) Saving to GGUF / merging to 16bit for vLLM
# (2) Continued training from a saved LoRA adapter
# (3) Adding an evaluation loop / OOMs
# (4) Customized chat templates